# 🧩 Prerequisites

In [ ]:
# @title 📀 Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
%cd /content/drive/MyDrive/Colab/thesis/rrs
CWD = Path.cwd()

In [ ]:
# @title 📦 Package Installation
%%capture
import os, re
!pip install -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !uv pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !uv pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

# 📊 Dataset

In [ ]:
from datasets import load_dataset, DatasetDict

train_dataset = load_dataset("json", data_files={
    "train": "data/train2-sft.jsonl",
}, split="train")

eval_dataset = load_dataset("json", data_files={
    "eval" : "data/dev-gen1.jsonl",
}, split="eval")
eval_dataset = eval_dataset.rename_column("biobart_gen", "hallcn")

dataset = DatasetDict({
    "train": train_dataset,
    "eval": eval_dataset,
})
print(dataset)

# 🚀 Train Model

In [ ]:
# @title Load Model
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

In [ ]:
# @title PEFT for LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
import random
import requests

random.seed(42)

word_site = "https://www.mit.edu/~ecprice/wordlist.10000"
response = requests.get(word_site)
words = response.text.splitlines()

def generate_random_sentence():
    num_words = random.randint(10, 15)
    random_words = random.sample(words, k=num_words)
    sentence = " ".join(random_words)
    sentence = sentence.capitalize() + "?"
    return sentence

In [ ]:
import random

random.seed(42)

def mix_hallucinations_batched(batch):
    new_summaries = []
    new_hallcns = []

    batch_size = len(batch["impression"])
    for i in range(batch_size):
        original_summary = batch["impression"][i]
        original_hallcn = batch["hallcn"][i]

        # 80% chance to be hallucinated
        if random.random() < 0.8:
            new_hallcns.append(original_hallcn)
            new_summaries.append(original_summary)

        # 20% chance to be random sentence
        else:
            new_hallcns.append(generate_random_sentence())
            new_summaries.append(original_summary)

    batch["impression"] = new_summaries
    batch["hallcn"] = new_hallcns

    return batch

# Apply the function using batched=True
dataset["train"] = dataset["train"].map(mix_hallucinations_batched, batched=True)

In [ ]:
# @title Chat Template Config
system_prompt = """You are a Medical Radiology Hallucination Detector and Corrector."""
user_prompt = """\
# Instructions:
1. Compare the [RADIOLOGY FINDINGS] with its [RADIOLOGY IMPRESSION].
2. Identify the incorrect detail in the [RADIOLOGY IMPRESSION] and fix it using the [RADIOLOGY FINDINGS].
3. Output ONLY the final [CORRECTED RADIOLOGY IMPRESSION] with minimal text.

# Context:
[RADIOLOGY FINDINGS]: {}
[RADIOLOGY IMPRESSION]: {}

# Output:
[CORRECTED RADIOLOGY IMPRESSION]:"""

# clean_pattern = re.compile(r"(?:\n|/\\|\\/|\s+)")

def generate_conversation(examples):
    sources = examples["findings"]
    summaries = examples["impression"]
    targets = examples["hallcn"]

    conversations = []
    for source, summary, target in zip(sources, summaries, targets):
        # source = clean_pattern.sub(" ", source).strip()
        conversations.append([
            {"role" : "system",    "content" : system_prompt},
            {"role" : "user",      "content" : user_prompt.format(source, target)},
            {"role" : "assistant", "content" : summary},
        ])
    return { "conversations": conversations, }

dataset = dataset.map(generate_conversation, batched = True)

In [ ]:
# @title Apply Chat Template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
        ) for convo in convos
    ]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from pprint import pprint

pprint(dataset["train"][0]["conversations"], width=160)
print(dataset["train"][0]["text"])

In [ ]:
# @title TRL Supervised Fine-Tuning Trainer and Config Setup
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["eval"],
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 10,
        output_dir="/content/outputs",
        eval_strategy="steps",
        save_strategy="no",
        eval_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        bf16=True, tf32=True,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
# model.save_pretrained_merged(
#     "models/qwen3-lora-sft",
#     tokenizer, save_method = "merged_16bit"
# )
model.save_pretrained("models/qwen3-sft-lora")
tokenizer.save_pretrained("models/qwen3-sft-lora")

# 🔥 Inference

In [ ]:
# @title
FastLanguageModel.for_inference(model)

system_prompt = """You are a helpful assistant. Compare the KNOWLEDGE with the QUESTION.
Identify the incorrect detail in the QUESTION and fix it using the KNOWLEDGE.
Output ONLY the corrected question in fewer than 15 words."""

user_prompt = "KNOWLEDGE: {}\n\nQUESTION: {}\n\nCORRECTED QUESTION:"

source = "23 surgeries and counting......lower lip birthmark, have tried all options out the there and guess what still have it, continues to grow back.....any suggestions? Is there a cure coming in the next few years hopefully?"
target = "What are the treatments for upper lip birthmark?"

messages = [
    {"role" : "system",    "content" : system_prompt},
    {"role" : "user",      "content" : user_prompt.format(source, target)},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 512, # Increase for longer outputs!
    use_cache = True,
    do_sample=False,
    temperature = 0.0, top_p = 1.0, top_k = 0,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "models/qwen3-sft-lora",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

In [ ]:
model.save_pretrained_merged(
    "/content/qwen3-sft-merged",
    tokenizer, save_method = "merged_16bit"
)

In [ ]:
!ls -lha /content/qwen3-sft-merged/

In [ ]:
!uv pip install ctranslate2

In [ ]:
!ct2-transformers-converter --model /content/qwen3-sft-merged --output_dir /content/drive/MyDrive/Colab/thesis/rrs/models/qwen3-sft-merged-ct2 --quantization int8_float16

In [ ]:
%cd /content/qwen3-sft-merged/

In [ ]:
!cp added_tokens.json chat_template.jinja special_tokens_map.json tokenizer.json tokenizer_config.json vocab.json /content/drive/MyDrive/Colab/thesis/rrs/models/qwen3-sft-merged-ct2/

In [ ]:
!ls -lh /content/drive/MyDrive/Colab/thesis/rrs/models/qwen3-sft-merged-ct2